# 03 · Detect (model family A + confident learning)

**What this step does.** For every corrupted training set:
1. **Out-of-fold probabilities** from logistic regression (family A). Each record's probability comes from a model that never saw that record.
2. **Noise estimate** (H1): confident learning estimates the joint distribution of given vs. true labels. The off-diagonal mass is the estimated share of wrong labels. We also keep the *missed-positive* cell (given 0, truly 1), which should grow under class-conditional and targeted noise.
3. **Record-level flags**, two ways (H3):
   - *global audit:* confident learning on the whole training set;
   - *stratum audit:* confident learning only inside the Stage 0 risk stratum, so its thresholds adapt to that subgroup.
4. **Flag quality** against the records we actually flipped: precision, recall, F1, globally and inside the stratum.
5. **Group-level rate check** (H3b): is the positive rate suspiciously **low**, overall and inside the stratum? It is compared with a **small independent audit sample** of the same group (400 re-verified records; 40 for South German Credit), drawn here from the clean validation split. In practice that sample would be records whose outcome was re-verified (e.g. linked to other hospitals) or a published rate. Using a small sample, not the full clean rate, keeps the check honest: the reference has its own sampling error. One-sided Fisher exact test. The global check gets an audit sample of the **same size**, so both checks cost the same review effort.
6. **Expert-review budget** (H3c): an expert can review 5% of the stratum. Which label-0 records should they check?
   - *ranked:* the ones the model thinks are most likely positive;
   - *random:* picked at random (what an auditor does with no model).

**Why steps 5-6 were added (Oct 9 trial run).** Confident learning found random flips well but almost none of the "positive recorded as negative" flips in Diabetes 130, because readmission is hard to predict and a hidden readmission looks like an ordinary non-readmission. A group-level check can still notice that a group has *too few* positives, and Stage 0 tells it which group to check. This is the paper's practical recommendation: diagnose where, check the group's rate, then spend expert review there.

**About the floor.** On clean data the detector already flags a share of records (about 0.19 in the September run). Those are hard or ambiguous cases, not proven errors. So for H1 we compare **estimate minus the dose-0 estimate** with the injected share.

**Inputs:** `results/noisy/*.csv`, `data/{dataset}_train.csv` · **Outputs:** `results/detect/{run}.csv` (per-record probability and flags), `results/03_detect_summary.csv`.

Runtime: roughly 5-10 s per run for Diabetes 130 and CDC on a laptop (105 runs each ≈ 10-20 min); South German Credit is fast. Set `QUICK = True` to run seeds 0-1 only.

**Which datasets run:** `li.RUN_ONLY` in `li_lib.py` (now: Taiwan credit only). Results for the other datasets are kept: the CSVs are merged, not overwritten.

In [1]:
# --- setup: run this cell first -------------------------------------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))          # li_lib.py sits next to the notebooks
import li_lib as li
pd.set_option("display.width", 160, "display.max_columns", 30)
print("li_lib", li.VERSION, "| project folder:", li.ROOT)
li.versions()

li_lib 1.1.0 | project folder: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity


,package,version
0,numpy,2.2.6
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,xgboost,3.4.1
4,cleanlab,2.9.0
5,lingam,1.13.0
6,causal-learn,0.1.4.8


In [2]:
QUICK = False
seeds = li.SEEDS[:2] if QUICK else li.SEEDS
(li.RESULTS / "detect").mkdir(exist_ok=True)
rows, t0 = [], time.time()
for name, cfg in li.run_configs().items():
    cfg = li.apply_frozen(cfg)
    tr = li.load_split(cfg, "train")
    X = li.Encoder(cfg).fit(tr).transform(tr).to_numpy(float)
    va = li.load_split(cfg, "val")                       # clean: source of the audit samples
    va_y, va_s = va[cfg.label].to_numpy(int), li.stratum_mask(va, cfg)
    m_audit = li.AUDIT_SIZE[name]
    for mech in li.MECHANISMS:
        for dose in li.DOSES:
            for seed in seeds:
                run = f"{name}__{mech}__{dose:.2f}__s{seed}"
                nz = pd.read_csv(li.RESULTS / "noisy" / f"{run}.csv")
                yn, fl, st = nz.y_noisy.to_numpy(int), nz.flipped.to_numpy(bool), nz.in_stratum.to_numpy(bool)
                P = li.oof_probabilities(X, yn, seed)
                est = li.estimated_noise(yn, P)
                est_s = li.estimated_noise(yn[st], P[st])
                fg = li.find_flags(yn, P)
                fs = li.find_flags(yn, P, within=st)
                fbr = li.budget_flags(yn, P[:, 1], st, how="ranked")
                fbx = li.budget_flags(yn, P[:, 1], st, how="random", seed=seed)
                pd.DataFrame({"row": nz.row, "p1": P[:, 1], "flag_global": fg, "flag_stratum": fs,
                              "flag_budget_ranked": fbr, "flag_budget_random": fbx}).to_csv(
                    li.RESULTS / "detect" / f"{run}.csv", index=False)
                rc_all = li.rate_check(yn, np.ones(len(yn), bool), li.audit_sample(va_y, m_audit, seed))
                rc_str = li.rate_check(yn, st, li.audit_sample(va_y[va_s], m_audit, seed))
                r = {"dataset": name, "mechanism": mech, "dose": dose, "seed": seed,
                     "realized_global_fraction": fl.mean(), "realized_stratum_fraction": fl[st].mean(),
                     "est_noise": est["noise_rate"], "est_missed_pos": est["est_given0_true1"],
                     "est_noise_stratum": est_s["noise_rate"], "est_missed_pos_stratum": est_s["est_given0_true1"],
                     "rate_global_observed": rc_all["observed_rate"], "rate_global_reference": rc_all["reference_rate"],
                     "rate_global_p": rc_all["p_value"],
                     "rate_stratum_observed": rc_str["observed_rate"], "rate_stratum_reference": rc_str["reference_rate"],
                     "rate_stratum_p": rc_str["p_value"], "rate_stratum_est_missing": rc_str["est_missing_pos"],
                     "true_missing_in_stratum": int((fl & st & (yn == 0)).sum())}
                for tag, flags, within in [("global", fg, None), ("global_in_stratum", fg, st), ("stratum", fs, st),
                                           ("budget_ranked", fbr, st), ("budget_random", fbx, st)]:
                    q = li.flag_quality(flags, fl, within)
                    r.update({f"{tag}_{k}": v for k, v in q.items()})
                rows.append(r)
        print(f"{name}: done ({time.time()-t0:.0f}s elapsed)")
summary = pd.DataFrame(rows)
summary = li.save_merged(summary, "03_detect_summary.csv", show=0)   # all datasets, for the cells below

taiwancredit: done (98s elapsed)
taiwancredit: done (194s elapsed)
taiwancredit: done (291s elapsed)
kept 315 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\03_detect_summary.csv  (420 rows x 53 cols)


In [3]:
base = summary[summary.dose == 0][["dataset", "mechanism", "seed", "est_noise", "est_noise_stratum"]]
h1 = summary.merge(base, on=["dataset", "mechanism", "seed"], suffixes=("", "_dose0"))
h1["est_above_floor"] = h1.est_noise - h1.est_noise_dose0
h1["est_above_floor_stratum"] = h1.est_noise_stratum - h1.est_noise_stratum_dose0
fits = []
for (d, m), g in h1.groupby(["dataset", "mechanism"]):
    x, y = g.realized_global_fraction, g.est_above_floor
    slope, icpt = np.polyfit(x, y, 1)
    fits.append({"dataset": d, "mechanism": m, "slope": slope, "intercept": icpt, "pearson_r": np.corrcoef(x, y)[0, 1]})
h1_fits = pd.DataFrame(fits)
li.save(h1_fits, "03_h1_dose_recovery.csv", show=0)
h1_fits.round(3)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\03_h1_dose_recovery.csv  (12 rows x 5 cols)


,dataset,mechanism,slope,intercept,pearson_r
0,cdcdiabetes,class_conditional,0.136,0.000,0.852
1,cdcdiabetes,targeted,-0.066,0.001,-0.157
2,cdcdiabetes,uniform,0.772,0.029,0.979
3,diabetes130,class_conditional,-0.086,-0.000,-0.218
4,diabetes130,targeted,-0.753,0.001,-0.747
5,diabetes130,uniform,0.647,0.033,0.970
6,southgerman,class_conditional,0.270,0.001,0.532
7,southgerman,targeted,0.504,0.002,0.317
8,southgerman,uniform,0.648,0.022,0.951
9,taiwancredit,class_conditional,0.045,0.001,0.504


In [4]:
cols = ["global_in_stratum_precision", "global_in_stratum_recall", "global_in_stratum_f1",
        "stratum_precision", "stratum_recall", "stratum_f1"]
summary[summary.mechanism == "targeted"].groupby(["dataset", "dose"])[cols].mean().round(3)

global_in_stratum_precision  global_in_stratum_recall  global_in_stratum_f1  stratum_precision  stratum_recall  stratum_f1
dataset      dose                                                                                                                            
cdcdiabetes  0.00                        0.000                       NaN                   NaN              0.000             NaN         NaN
             0.03                        0.013                     0.167                 0.025              0.013           0.167       0.025
             0.05                        0.023                     0.167                 0.040              0.023           0.167       0.040
             0.10                        0.043                     0.153                 0.067              0.044           0.153       0.069
             0.20                        0.059                     0.108                 0.076              0.066           0.108       0.082
             0.30                        0.074                     0.087                 0.080              0.091           0.087       0.089
             0.40                        0.074                     0.059                 0.066              0.096           0.059       0.073
diabetes130  0.00                        0.000                       NaN                   NaN              0.000             NaN         NaN
             0.03                        0.000                     0.000                 0.000              0.000           0.000       0.000
             0.05                        0.001                     0.005                 0.001              0.000           0.005       0.001
             0.10                        0.001                     0.002                 0.001              0.000           0.002       0.001
             0.20                        0.002                     0.005                 0.003              0.002           0.005       0.003
             0.30                        0.002                     0.002                 0.002              0.002           0.002       0.002
             0.40                        0.002                     0.002                 0.002              0.002           0.002       0.002
southgerman  0.00                        0.000                       NaN                   NaN              0.000             NaN         NaN
             0.03                        0.006                     0.200                 0.012              0.005           0.200       0.010
             0.05                        0.020                     0.300                 0.037              0.017           0.300       0.031
             0.10                        0.082                     0.520                 0.142              0.073           0.520       0.127
             0.20                        0.116                     0.340                 0.173              0.105           0.340       0.160
             0.30                        0.163                     0.371                 0.226              0.149           0.371       0.212
             0.40                        0.210                     0.326                 0.254              0.195           0.326       0.243
taiwancredit 0.00                        0.000                       NaN                   NaN              0.000             NaN         NaN
             0.03                        0.027                     0.312                 0.049              0.017           0.312       0.033
             0.05                        0.043                     0.304                 0.075              0.028           0.304       0.052
             0.10                        0.073                     0.257                 0.114              0.049           0.257       0.083
             0.20                        0.111                     0.178                 0.137              0.077           0.178       0.108
             0.30                        0.

In [5]:
a = li.RATE_CHECK_ALPHA
pw = (summary.assign(global_fires=summary.rate_global_p < a, stratum_fires=summary.rate_stratum_p < a)
      .groupby(["dataset", "mechanism", "dose"])[["global_fires", "stratum_fires"]].mean())
li.save(pw.reset_index(), "03_h3b_rate_check_power.csv", show=0)
pw.unstack("mechanism").round(2)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\03_h3b_rate_check_power.csv  (84 rows x 5 cols)


global_fires                      stratum_fires                 
mechanism         class_conditional targeted uniform class_conditional targeted uniform
dataset      dose                                                                      
cdcdiabetes  0.00               0.0      0.0     0.0               0.0      0.0     0.0
             0.03               0.0      0.0     0.0               0.0      0.0     0.0
             0.05               0.0      0.0     0.0               0.0      0.0     0.0
             0.10               0.2      0.0     0.0               0.2      0.2     0.0
             0.20               0.8      0.0     0.0               0.6      0.8     0.0
             0.30               1.0      0.0     0.0               0.8      0.8     0.0
             0.40               1.0      0.0     0.0               1.0      1.0     0.0
diabetes130  0.00               0.2      0.2     0.2               0.0      0.0     0.0
             0.03               0.2      0.2     0.0               0.0      0.0     0.0
             0.05               0.2      0.2     0.0               0.0      0.0     0.0
             0.10               0.2      0.2     0.0               0.0      0.0     0.0
             0.20               0.6      0.2     0.0               0.2      0.0     0.0
             0.30               0.6      0.2     0.0               0.6      0.6     0.0
             0.40               0.6      0.2     0.0               0.8      0.8     0.0
southgerman  0.00               0.0      0.0     0.0               0.0      0.0     0.0
             0.03               0.0      0.0     0.0               0.0      0.0     0.0
             0.05               0.0      0.0     0.0               0.0      0.0     0.0
             0.10               0.0      0.0     0.0               0.0      0.0     0.0
             0.20               0.2      0.0     0.0               0.0      0.0     0.0
             0.30               0.8      0.0     0.0               0.2      0.0     0.0
             0.40               1.0      0.0     0.0               0.8      1.0     0.0
taiwancredit 0.00               0.0      0.0     0.0               0.2      0.2     0.2
             0.03               0.0      0.0     0.0               0.2      0.2     0.2
             0.05               0.0      0.0     0.0               0.2      0.2     0.0
             0.10               0.0      0.0     0.0               0.8      0.8     0.0
             0.20               0.6      0.0     0.0               1.0      1.0     0.0
             0.30               1.0      0.0     0.0               1.0      1.0     0.0
             0.40               1.0      0.2     0.0               1.0      1.0     0.0

## H3c quick check: where should the expert's review budget go?
Precision = share of reviewed records that really were flipped. **Look for:** ranked review above random review. If both are low, the honest conclusion is that individual missed events can't be identified from these features; the rate check above can still size the problem.

In [6]:
cols = ["budget_ranked_precision", "budget_ranked_recall", "budget_random_precision", "budget_random_recall"]
summary[summary.mechanism.isin(["targeted", "class_conditional"])].groupby(["dataset", "mechanism", "dose"])[cols].mean().round(3)

budget_ranked_precision  budget_ranked_recall  budget_random_precision  budget_random_recall
dataset      mechanism         dose                                                                                              
cdcdiabetes  class_conditional 0.00                    0.000                   NaN                    0.000                   NaN
                               0.03                    0.027                 0.254                    0.002                 0.021
                               0.05                    0.053                 0.373                    0.007                 0.050
                               0.10                    0.088                 0.285                    0.020                 0.062
                               0.20                    0.181                 0.288                    0.024                 0.039
                               0.30                    0.233                 0.253                    0.038                 0.041
                               0.40                    0.314                 0.251                    0.072                 0.057
             targeted          0.00                    0.000                   NaN                    0.000                   NaN
                               0.03                    0.024                 0.256                    0.007                 0.078
                               0.05                    0.053                 0.333                    0.015                 0.093
                               0.10                    0.097                 0.307                    0.025                 0.080
                               0.20                    0.171                 0.272                    0.042                 0.067
                               0.30                    0.240                 0.256                    0.074                 0.079
                               0.40                    0.302                 0.241                    0.065                 0.052
diabetes130  class_conditional 0.00                    0.000                   NaN                    0.000                   NaN
                               0.03                    0.013                 0.208                    0.006                 0.099
                               0.05                    0.017                 0.166                    0.006                 0.057
                               0.10                    0.032                 0.156                    0.010                 0.052
                               0.20                    0.067                 0.153                    0.023                 0.052
                               0.30                    0.095                 0.153                    0.034                 0.054
                               0.40                    0.118                 0.138                    0.048                 0.056
             targeted          0.00                    0.000                   NaN                    0.000                   NaN
                               0.03                    0.014                 0.216                    0.003                 0.040
                               0.05                    0.019                 0.180                    0.007                 0.063
                               0.10                    0.031                 0.144                    0.013                 0.063
                               0.20                    0.068                 0.160                    0.027                 0.063
                               0.30                    0.088                 0.137                    0.036                 0.057
                               0.40                    0.134                 0.157                    0.055                 0.065
southgerman  class_conditional 0.00                    0.000                   NaN                    0.000                   NaN
       

## What to look for before moving on
- [ ] H1: positive slope and high correlation for uniform and class-conditional noise.
- [ ] H1: under targeted noise, the *global* estimate barely moves, while the *stratum* estimate rises. That contrast is H2's mechanism.
- [ ] H3: compare the two audits inside the stratum.
- [ ] H3b: the stratum rate check's power under targeted noise vs. the global check's.
- [ ] H3c: ranked vs. random review precision.